# ResNet18 Performance comparison between PyTorch (fp32), TensorRT (fp16), and TensorRT (int8)

In [ ]:
import os
import time
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset

import tensorrt as trt
from IPython.display import display

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = "imagenet-mini-subset/val"   # ImageFolder 구조
PTH_PATH = "models/resnet18.pth"
FP16_ENGINE_PATH = "engines/resnet18_fp16.plan"
INT8_ENGINE_PATH = "engines/resnet18_int8.plan"

NUM_SAMPLES = 100
BATCH_SIZE = 1
NUM_CLASSES = 1000
INPUT_SHAPE = (1, 3, 224, 224)

In [ ]:
# 이미지 전처리
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = ImageFolder(DATA_DIR, transform=transform)

indices = list(range(min(NUM_SAMPLES, len(dataset))))
subset = Subset(dataset, indices)

# dataloader 생성
loader = DataLoader(
    subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("dataset size:", len(subset))
print("num classes:", len(dataset.classes))

## Load PyTorch model

In [ ]:
model_torch = models.resnet18(weights=None)
model_torch.load_state_dict(torch.load(PTH_PATH, map_location=DEVICE))
model_torch.eval().to(DEVICE)

print("Loaded PyTorch(fp32) model")

## Load TensorRT model

In [ ]:
TRT_LOGGER = trt.Logger(trt.Logger.WARNING)

def load_engine(engine_path):
    with open(engine_path, "rb") as f, trt.Runtime(TRT_LOGGER) as runtime:
        engine = runtime.deserialize_cuda_engine(f.read())
    return engine

def allocate_buffers(engine):
    context = engine.create_execution_context()

    input_name = engine.get_tensor_name(0)
    output_name = engine.get_tensor_name(1)

    input_shape = tuple(engine.get_tensor_shape(input_name))
    output_shape = tuple(engine.get_tensor_shape(output_name))

    input_dtype = trt.nptype(engine.get_tensor_dtype(input_name))
    output_dtype = trt.nptype(engine.get_tensor_dtype(output_name))

    # PyTorch를 사용하여 GPU 메모리 할당
    d_input = torch.empty(input_shape, dtype=torch.from_numpy(np.empty(0, dtype=input_dtype)).dtype).cuda()
    d_output = torch.empty(output_shape, dtype=torch.from_numpy(np.empty(0, dtype=output_dtype)).dtype).cuda()

    # TensorRT context에 메모리 주소(data_ptr) 전달
    context.set_tensor_address(input_name, d_input.data_ptr())
    context.set_tensor_address(output_name, d_output.data_ptr())

    # PyTorch의 현재 스트림 사용
    stream = torch.cuda.current_stream()

    return {
        "context": context,
        "input_name": input_name,
        "output_name": output_name,
        "input_shape": input_shape,
        "output_shape": output_shape,
        "input_dtype": input_dtype,
        "output_dtype": output_dtype,
        "d_input": d_input,
        "d_output": d_output,
        "stream": stream,
    }

In [ ]:
def trt_infer(buffers, x_np):
    context = buffers["context"]
    
    # 입력을 GPU로 복사 (numpy -> torch tensor)
    # x_np가 이미 tensor라면 .copy_()만 하면 됩니다.
    x_tensor = torch.from_numpy(x_np).to(device="cuda", dtype=buffers["d_input"].dtype)
    buffers["d_input"].copy_(x_tensor)

    # 추론 실행 (스트림 핸들 전달)
    context.execute_async_v3(stream_handle=buffers["stream"].cuda_stream)
    
    # 결과를 다시 numpy로 변환하여 반환
    # (결과가 GPU에 있으므로 다시 CPU로 가져와야 함)
    output = buffers["d_output"].cpu().numpy()
    
    return output

In [ ]:
import json
import urllib.request

# ImageNet 공식 인덱스 정보 다운로드
url = "https://raw.githubusercontent.com/raghakot/keras-vis/master/resources/imagenet_class_index.json"
with urllib.request.urlopen(url) as response:
    class_index = json.loads(response.read().decode())

# {'n01443537': 0, ...} 형태의 딕셔너리 생성
wnid_to_idx = {v[0]: int(k) for k, v in class_index.items()}

def evaluate_torch(model, loader):
    correct = 0
    total = 0
    times = []

    dataset_idx_to_imagenet_idx = np.array([
        wnid_to_idx[cls_name] for cls_name in loader.dataset.dataset.classes
    ])

    with torch.no_grad():
        # warmup
        x_warm = torch.randn(*INPUT_SHAPE).to(DEVICE)
        for _ in range(20):
            _ = model(x_warm)
        if DEVICE == "cuda":
            torch.cuda.synchronize()

        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            start = time.time()
            outputs = model(images)
            if DEVICE == "cuda":
                torch.cuda.synchronize()
            end = time.time()

            preds = outputs.argmax(dim=1).cpu().numpy()

            real_labels = dataset_idx_to_imagenet_idx[labels.cpu().numpy()]

            correct += (preds == real_labels).sum().item()
            total += labels.size(0)
            times.append((end - start) * 1000)

    avg_ms = sum(times) / len(times)
    fps = 1000.0 / avg_ms
    acc = correct / total

    return {
        "backend": "torch",
        "precision": "fp32",
        "avg_latency_ms": avg_ms,
        "fps": fps,
        "top1_acc": acc
    }

def evaluate_trt(engine_path, loader, precision):
    engine = load_engine(engine_path)
    buffers = allocate_buffers(engine)

    correct = 0
    total = 0
    times = []

    dataset_idx_to_imagenet_idx = np.array([
        wnid_to_idx[cls_name] for cls_name in loader.dataset.dataset.classes
    ])

    # warmup
    x_warm = np.random.randn(*buffers["input_shape"]).astype(np.float32)
    for _ in range(20):
        _ = trt_infer(buffers, x_warm)

    for images, labels in loader:
        x_np = images.numpy()

        start = time.time()
        outputs = trt_infer(buffers, x_np)
        end = time.time()

        preds = outputs.argmax(axis=1)
        
        real_labels = dataset_idx_to_imagenet_idx[labels.cpu().numpy()]

        correct += (preds == real_labels).sum()
        total += labels.size(0)
        times.append((end - start) * 1000)

    avg_ms = np.mean(times)
    fps = 1000.0 / avg_ms
    acc = correct / total

    return {
        "backend": "tensorrt",
        "precision": precision,
        "avg_latency_ms": avg_ms,
        "fps": fps,
        "top1_acc": acc
    }

In [ ]:
results = []

torch_result = evaluate_torch(model_torch, loader)
results.append(torch_result)
print(torch_result)

fp16_result = evaluate_trt(FP16_ENGINE_PATH, loader, "fp16")
results.append(fp16_result)
print(fp16_result)

int8_result = evaluate_trt(INT8_ENGINE_PATH, loader, "int8")
results.append(int8_result)
print(int8_result)

In [ ]:
df = pd.DataFrame(results)
df["avg_latency_ms"] = df["avg_latency_ms"].round(3)
df["fps"] = df["fps"].round(2)
df["top1_acc"] = df["top1_acc"].round(4)

display(df)

In [ ]:
torch_latency = df[df["backend"] == "torch"]["avg_latency_ms"].values[0]

df["speedup_vs_torch"] = (torch_latency / df["avg_latency_ms"]).round(2)
display(df)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

# 오류 시 matplotlib upgrade 후 다시 실행
# !pip3 install --upgrade matplotlib

df["label"] = df["backend"] + "_" + df["precision"]

plt.figure(figsize=(6,4))
plt.bar(df["label"], df["avg_latency_ms"])
plt.ylabel("Latency (ms)")
plt.title("Inference Latency Comparison")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6,4))
plt.bar(df["label"], df["top1_acc"])
plt.ylabel("Top-1 Accuracy")
plt.title("Accuracy Comparison")
plt.ylim(0.65, 0.73)
plt.tight_layout()
plt.show()